# Tabela de candidatos por série — visão de desenvolvimento (ticket #6)

**Dados:** somente a visão de desenvolvimento (via `abrir_desenvolvimento`). O período OOT não é lido por nenhuma célula (ADR 0002). **Nenhum modelo é treinado ou ajustado aqui**: o treino começa no #7.

**Plano congelado antes de qualquer resultado:** `docs/analysis/plano-candidatos.md` (decisão D-004, aprovada). Direção: D-003 (aprovada).

**Status:** tabelas geradas a partir de `case_mr_health.candidatos`. Observações e Interpretações abaixo ficam **PROPOSTAS** até a aprovação do usuário.

**Regra de entrada:** um candidato só é excluído por (a) premissa estrutural refutada por teste do #5, (b) inviabilidade aritmética ou (c) decisão do usuário. Ausência de evidência não exclui: o candidato entra rotulado como **hipótese**.

**Limitações de amostra:** 76 a 79 dias (diária) e 10 semanas (semanal); testes do #5 exploratórios; estacionariedade e autocorrelação semanais não testáveis; sazonalidade de A e D inconclusiva por baixo poder.

## Perguntas
1. A evidência do #5 usada pela tabela se reproduz na visão de desenvolvimento?
2. Quais candidatos entram em cada uma das 8 séries, e com que base (evidência, hipótese, decisão do usuário ou aritmética)?
3. Quais são as premissas de cada candidato e a EDA do #5 as atende?
4. O espaço de hiperparâmetros respeita o teto de 4 combinações?
5. Que requisito de backtest vale para o item C semanal?

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

from case_mr_health.candidatos import EVIDENCIAS, SKUS, candidatos, tabela
from case_mr_health.config import Settings
from case_mr_health.diagnosticos import (
    autocorrelacao,
    estacionariedade,
    intermitencia,
    sazonalidade_semanal,
)
from case_mr_health.particoes import abrir_desenvolvimento

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)
SEED = 20211030
ALFA = 0.05
MIN_TESTES = 2  # critério que reproduz a leitura aprovada no #5

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'pyproject.toml').is_file()
)
cfg = Settings(
    desenvolvimento_db_path=ROOT / 'data/processed/desenvolvimento.duckdb'
)
con = abrir_desenvolvimento(cfg)  # única fonte de dados deste notebook

d = con.sql(
    'SELECT sku, data, quantidade FROM serie_diaria ORDER BY sku, data'
).df()
w = con.sql(
    'SELECT sku, semana, quantidade FROM serie_semanal ORDER BY sku, semana'
).df()
part = con.sql('SELECT * FROM particao ORDER BY granularidade, sku').df()
folds = con.sql('SELECT * FROM fold ORDER BY sku, granularidade, fold').df()
d['data'] = pd.to_datetime(d['data'])
w['semana'] = pd.to_datetime(w['semana'])
skus = sorted(d['sku'].unique())
assert tuple(skus) == SKUS
ydia = {s: d.loc[d['sku'] == s, 'quantidade'].to_numpy(float) for s in skus}
ysem = {s: w.loc[w['sku'] == s, 'quantidade'].to_numpy(float) for s in skus}

limite = part.set_index(['sku', 'granularidade'])['desenvolvimento_fim']
for s in skus:
    assert d.loc[d.sku == s, 'data'].max() <= pd.Timestamp(limite[s, 'diaria'])
    assert w.loc[w.sku == s, 'semana'].max() <= pd.Timestamp(
        limite[s, 'semanal']
    )
display(part)

,sku,granularidade,desenvolvimento_inicio,desenvolvimento_fim,oot_inicio,oot_fim
0,item A,diaria,2021-06-01,2021-08-18,2021-08-19,2021-09-01
1,item B,diaria,2021-06-04,2021-08-18,2021-08-19,2021-09-01
2,item C,diaria,2021-06-01,2021-08-18,2021-08-19,2021-09-01
3,item D,diaria,2021-06-04,2021-08-18,2021-08-19,2021-09-01
4,item A,semanal,2021-06-07,2021-08-15,2021-08-16,2021-08-29
5,item B,semanal,2021-06-07,2021-08-15,2021-08-16,2021-08-29
6,item C,semanal,2021-06-07,2021-08-15,2021-08-16,2021-08-29
7,item D,semanal,2021-06-07,2021-08-15,2021-08-16,2021-08-29


## 1. Confronto da evidência do #5 com o módulo

A tabela usa os resultados do #5 como entrada fixa (`EVIDENCIAS`). Aqui eles são **recalculados na visão de desenvolvimento** e conferidos; qualquer divergência interrompe o notebook.

- **Intermitência** (Syntetos, Boylan e Croston, 2005): classe por ADI e CV².
- **Estacionariedade**: ADF (H0: raiz unitária) e KPSS (H0: estacionária em nível), só com n >= 30.
- **Autocorrelação**: Ljung–Box nos lags 7 e 14 (diária); H0: ruído.
- **Sazonalidade por dia da semana**: Kruskal–Wallis, Friedman e permutação nas 10 semanas completas (06-07 a 08-15), com Holm nos 4 SKUs. O rótulo **"confirmada"** reproduz a classificação aprovada no #5: pelo menos 2 dos 3 testes com p (Holm) < 0,05. Não é um teste novo; é a regra que reproduz a leitura aprovada, e os p são exploratórios.

In [2]:
semanas = sorted(w['semana'].unique())


def matriz(df_sku):
    x = df_sku.set_index('data')['quantidade']
    return np.array(
        [
            [x[pd.Timestamp(wk) + pd.Timedelta(days=i)] for i in range(7)]
            for wk in semanas
        ],
        float,
    )


mats = {s: matriz(d[d.sku == s]) for s in skus}
res = {s: sazonalidade_semanal(m, seed=SEED) for s, m in mats.items()}
tab_saz = pd.DataFrame(res).T
for col in ('kruskal_p', 'friedman_p', 'permutacao_p'):
    tab_saz[col + '_holm'] = multipletests(
        tab_saz[col].fillna(1.0), method='holm'
    )[1]
tab_saz['testes_p<0,05'] = (
    tab_saz[[c for c in tab_saz if c.endswith('_holm')]] < ALFA
).sum(axis=1)
tab_saz['sazonalidade'] = np.where(
    tab_saz['testes_p<0,05'] >= MIN_TESTES, 'confirmada', 'inconclusiva'
)

linhas = []
for s in skus:
    inter_d = intermitencia(ydia[s])
    inter_s = intermitencia(ysem[s])
    est_d = estacionariedade(ydia[s])
    est_s = estacionariedade(ysem[s])
    lb = autocorrelacao(ydia[s], nlags=21, lags_lb=[7, 14])['ljung_box_p']
    linhas.append(
        dict(
            sku=s,
            zeros_diaria=f'{(ydia[s] == 0).mean():.1%}',
            classe_diaria=inter_d.classe,
            adi=round(inter_d.adi, 2),
            cv2=round(inter_d.cv2, 2),
            classe_semanal=inter_s.classe,
            estacionaria_diaria=est_d['veredito'],
            estacionaria_semanal=(
                'não testável' if not est_s['testavel'] else est_s['veredito']
            ),
            ljung_box_p_min=round(min(lb.values()), 2),
            sazonalidade_diaria=tab_saz.loc[s, 'sazonalidade'],
        )
    )
    ev_d = EVIDENCIAS[s, 'diaria']
    ev_s = EVIDENCIAS[s, 'semanal']
    assert inter_d.classe == ev_d.classe
    assert inter_s.classe == ev_s.classe
    assert est_d['veredito'] == 'estacionária'
    assert ev_d.estacionaria == 'sim'
    assert not est_s['testavel']
    assert ev_d.autocorrelacao_fraca == (min(lb.values()) > ALFA)
    assert tab_saz.loc[s, 'sazonalidade'] == ev_d.sazonalidade
conf = pd.DataFrame(linhas)
display(conf)
display(tab_saz.round(4))
print('ok: evidência do módulo = recálculo na visão de desenvolvimento')

,sku,zeros_diaria,classe_diaria,adi,cv2,classe_semanal,estacionaria_diaria,estacionaria_semanal,ljung_box_p_min,sazonalidade_diaria
0,item A,43.0%,intermitente,1.76,0.20,suave,estacionária,não testável,0.16,inconclusiva
1,item B,36.8%,intermitente,1.58,0.40,suave,estacionária,não testável,0.39,confirmada
2,item C,45.6%,intermitente,1.84,0.30,suave,estacionária,não testável,0.14,confirmada
3,item D,32.9%,intermitente,1.49,0.33,suave,estacionária,não testável,0.22,inconclusiva


,kruskal_p,friedman_p,permutacao_p,kruskal_p_holm,friedman_p_holm,permutacao_p_holm,"testes_p<0,05",sazonalidade
item A,0.0301,0.1000,0.0517,0.0601,0.1999,0.1034,0,inconclusiva
item B,0.0034,0.0014,0.0035,0.0134,0.0056,0.0140,3,confirmada
item C,0.0170,0.0144,0.0084,0.0510,0.0432,0.0252,2,confirmada
item D,0.1275,0.1549,0.0591,0.1275,0.1999,0.1034,0,inconclusiva


ok: evidência do módulo = recálculo na visão de desenvolvimento


**Observação (proposta, aguardando aprovação):** o recálculo na visão de desenvolvimento reproduz a evidência do #5 usada pelo módulo: diárias intermitentes, semanais suaves, diárias estacionárias em nível, Ljung–Box sem rejeitar ruído e sazonalidade por dia da semana confirmada em B e C e inconclusiva em A e D.

**Interpretação (proposta):** a tabela de candidatos pode usar essa evidência como entrada. Não há teste novo; os mesmos limites de amostra do #5 valem.

## 2. Tabela de candidatos por série

Uma tabela por série (4 SKUs × diária e semanal). Colunas: se entra; **base** da decisão (`evidencia` = teste do #5; `hipotese` = sem teste formal que sustente; `decisao_usuario`; `aritmetica`; `regra` = regra do ADR 0003); motivo; premissas com a situação (atendida, não atendida, inconclusiva, não testável) e a base de cada uma; espaço de hiperparâmetros e número de combinações (teto de 4).

In [3]:
def tabela_serie(sku, gran):
    linhas = []
    for c in candidatos(sku, gran):
        linhas.append({
            'algoritmo': c.algoritmo,
            'papel': c.papel,
            'entra': 'sim' if c.entra else 'NÃO',
            'base': c.base,
            'motivo': c.motivo,
            'premissas': '\n'.join(
                f'- {p.texto}: {p.situacao} ({p.base})' for p in c.premissas
            ),
            'espaço': c.espaco or '',
            'comb.': c.n_combinacoes,
            'observação': c.observacao,
        })
    return pd.DataFrame(linhas)


for gran in ('diaria', 'semanal'):
    for s in skus:
        print(f'### {s} — {gran}')
        display(tabela_serie(s, gran))

### item A — diaria


,algoritmo,papel,entra,base,motivo,premissas,espaço,comb.,observação
0,naive,baseline,sim,regra,Baseline obrigatório (ADR 0003): último valor observado.,- O último período é bom guia do próximo: nao_testavel (hipotese),,1,Na diária intermitente prevê zero depois de um dia sem venda; serve de piso de comparação.
1,media_historica,baseline,sim,regra,Baseline obrigatório (ADR 0003): média expansiva do treino.,- Nível aproximadamente constante (estacionária em nível): atendida (evidencia),,1,
2,naive_sazonal,baseline,sim,hipotese,Repete o mesmo dia da semana anterior (lag 7). Teste de sazonalidade inconclusivo (baixo poder); entra como hipótese por decisão do usuário.,- Sazonalidade por dia da semana (período 7): inconclusiva (hipotese),"{'periodo': (7,)}",1,
3,ets_ses,candidato,sim,regra,"ETS sem tendência nem sazonalidade (ANN), entre o naive e a média.","- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): nao_atendida (hipotese)","{'alfa': ('estimado',)}",1,
4,ets_holt_winters_aditivo,candidato,sim,hipotese,Nível e sazonalidade aditivos de período 7. Sem componente multiplicativo (zeros) e sem tendência (estacionária em nível).,"- Sazonalidade por dia da semana (período 7): inconclusiva (hipotese)\n- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): nao_atendida (hipotese)","{'periodo': (7,), 'tendencia': ('nenhuma',)}",1,
5,arima,candidato,sim,regra,Contraste não sazonal: d = 0 apoiado pelo ADF/KPSS. A autocorrelação fraca sugere pouco a modelar (hipótese: perto da média).,- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Dependência temporal a modelar (autocorrelação): nao_atendida (evidencia),"{'ordem': ((1, 0, 0), (0, 0, 1), (1, 0, 1))}",3,Com constante.
6,sarima,candidato,sim,hipotese,"ARIMA com termo sazonal AR(1) de período 7. O sazonal (0,0,0,7) coincide com o ARIMA sem sazonal e não é repetido.",- Sazonalidade por dia da semana (período 7): inconclusiva (hipotese)\n- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Dependência temporal a modelar (autocorrelação): nao_atendida (evidencia),"{'ordem': ((0, 0, 0), (1, 0, 0), (0, 0, 1)), 'ordem_sazonal': ((1, 0, 0, 7),)}",3,Com constante.
7,croston,candidato,sim,evidencia,Croston (1972): tamanho ÷ intervalo. Entra: a série é intermitente.,"- Demanda intermitente (ADI >= 1,32): atendida (evidencia)\n- Sem sazonalidade nem dia da semana (o método ignora): inconclusiva (hipotese)","{'alfa': ('0.1', '0.2', '0.3')}",3,"Implementação própria (ADR 0003). Previsão constante, sem dia da semana."
8,sba,candidato,sim,evidencia,Croston com correção de viés (Syntetos–Boylan). Entra: a série é intermitente.,"- Demanda intermitente (ADI >= 1,32): atendida (evidencia)\n- Sem sazonalidade nem dia da semana (o método ignora): inconclusiva (hipotese)","{'alfa': ('0.1', '0.2', '0.3')}",3,"Implementação própria (ADR 0003). Previsão constante, sem dia da semana."
9,tsb,candidato,sim,evidencia,Probabilidade de venda atualizada também nos zeros. Entra: a série é intermitente.,"- Demanda intermitente (ADI >= 1,32): atendida (evidencia)\n- Sem sazonalidade nem dia da semana (o método ignora): inconclusiva (hipotese)","{'alfa_beta': ('0.1', '0.2', '0.3')}",3,"Implementação própria (ADR 0003). Previsão constante, sem dia da semana. α = β no TSB."


### item B — diaria


,algoritmo,papel,entra,base,motivo,premissas,espaço,comb.,observação
0,naive,baseline,sim,regra,Baseline obrigatório (ADR 0003): último valor observado.,- O último período é bom guia do próximo: nao_testavel (hipotese),,1,Na diária intermitente prevê zero depois de um dia sem venda; serve de piso de comparação.
1,media_historica,baseline,sim,regra,Baseline obrigatório (ADR 0003): média expansiva do treino.,- Nível aproximadamente constante (estacionária em nível): atendida (evidencia),,1,
2,naive_sazonal,baseline,sim,evidencia,Repete o mesmo dia da semana anterior (lag 7). Sazonalidade confirmada por teste formal.,- Sazonalidade por dia da semana (período 7): atendida (evidencia),"{'periodo': (7,)}",1,
3,ets_ses,candidato,sim,regra,"ETS sem tendência nem sazonalidade (ANN), entre o naive e a média.","- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): nao_atendida (hipotese)","{'alfa': ('estimado',)}",1,
4,ets_holt_winters_aditivo,candidato,sim,evidencia,Nível e sazonalidade aditivos de período 7. Sem componente multiplicativo (zeros) e sem tendência (estacionária em nível).,"- Sazonalidade por dia da semana (período 7): atendida (evidencia)\n- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): nao_atendida (hipotese)","{'periodo': (7,), 'tendencia': ('nenhuma',)}",1,
5,arima,candidato,sim,regra,Contraste não sazonal: d = 0 apoiado pelo ADF/KPSS. A autocorrelação fraca sugere pouco a modelar (hipótese: perto da média).,- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Dependência temporal a modelar (autocorrelação): nao_atendida (evidencia),"{'ordem': ((1, 0, 0), (0, 0, 1), (1, 0, 1))}",3,Com constante.
6,sarima,candidato,sim,evidencia,"ARIMA com termo sazonal AR(1) de período 7. O sazonal (0,0,0,7) coincide com o ARIMA sem sazonal e não é repetido.",- Sazonalidade por dia da semana (período 7): atendida (evidencia)\n- Nível aproximadamente constante (estacionária em nível): atendida (evidencia)\n- Dependência temporal a modelar (autocorrelação): nao_atendida (evidencia),"{'ordem': ((0, 0, 0), (1, 0, 0), (0, 0, 1)), 'ordem_sazonal': ((1, 0, 0, 7),)}",3,Com constante.
7,croston,candidato,sim,evidencia,Croston (1972): tamanho ÷ intervalo. Entra: a série é intermitente.,"- Demanda intermitente (ADI >= 1,32): atendida (evidencia)\n- Sem sazonalidade nem dia da semana (o método ignora): nao_atendida (evidencia)","{'alfa': ('0.1', '0.2', '0.3')}",3,"Implementação própria (ADR 0003). Previsão constante, sem dia da semana."
8,sba,candidato,sim,evidencia,Croston com correção de viés (Syntetos–Boylan). Entra: a série é intermitente.,"- Demanda intermitente (ADI >= 1,32): atendida (evidencia)\n- Sem sazonalidade nem dia da semana (o método ignora): nao_atendida (evidencia)","{'alfa': ('0.1', '0.2', '0.3')}",3,"Implementação própria (ADR 0003). Previsão constante, sem dia da semana."
9,tsb,candidato,sim,evidencia,Probabilidade de venda atualizada também nos zeros. Entra: a série é intermitente.,"- Demanda intermitente (ADI >= 1,32): atendida (evidencia)\n- Sem sazonalidade nem dia da semana (o método ignora): nao_atendida (evidencia)","{'alfa_beta': ('0.1', '0.2', '0.3')}",3,"Implementação própria (ADR 0003). Previsão constante, sem dia da semana. α = β no TSB."


### item C — diaria


...

,algoritmo,papel,entra,base,motivo,premissas,espaço,comb.,observação
0,naive,baseline,sim,regra,Baseline obrigatório (ADR 0003): último valor observado.,- O último período é bom guia do próximo: nao_testavel (hipotese),,1,
1,media_historica,baseline,sim,regra,Baseline obrigatório (ADR 0003): média expansiva do treino.,- Nível aproximadamente constante (estacionária em nível): nao_testavel (hipotese),,1,
2,naive_sazonal,baseline,NÃO,aritmetica,Sem sazonalidade testável em 10 semanas; ciclo de 4 ou 52 semanas não cabe nos dados.,,,0,
3,ets_ses,candidato,sim,regra,"ETS sem tendência nem sazonalidade (ANN), entre o naive e a média.","- Nível aproximadamente constante (estacionária em nível): nao_testavel (hipotese)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): inconclusiva (hipotese)","{'alfa': ('estimado',)}",1,
4,ets_holt_amortecido,candidato,sim,regra,Tendência amortecida (AAdN); não há teste de tendência com 10 pontos.,"- Possível tendência amortecida: nao_testavel (hipotese)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): inconclusiva (hipotese)","{'phi': ('estimado em [0,8; 0,98]',)}",1,
5,arima,candidato,sim,decisao_usuario,"ARIMA mínimo incluído por decisão do usuário, para contraste com as demais opções. Base fraca: estacionariedade e autocorrelação semanais não são testáveis (n = 10 < 30); com 6 pontos no primeiro fold, até 3 parâmetros.",- Estacionariedade (escolha de d): nao_testavel (hipotese)\n- Dependência temporal a modelar: nao_testavel (hipotese),"{'ordem': ((1, 0, 0), (0, 1, 1))}",2,"(1,0,0) com constante; (0,1,1) sem constante. (0,1,1) é próximo do SES. Item C: variante opcional com variável de intervenção na semana 07-19."
6,sarima,candidato,NÃO,aritmetica,"Ciclo sazonal de 4 ou 52 semanas não cabe em 10 pontos. Exclusão sobre o universo do ADR 0003, sinalizada.",,,0,
7,croston,candidato,NÃO,evidencia,"Croston (1972): tamanho ÷ intervalo. Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,
8,sba,candidato,NÃO,evidencia,"Croston com correção de viés (Syntetos–Boylan). Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,
9,tsb,candidato,NÃO,evidencia,"Probabilidade de venda atualizada também nos zeros. Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,


### item C — semanal


,algoritmo,papel,entra,base,motivo,premissas,espaço,comb.,observação
0,naive,baseline,sim,regra,Baseline obrigatório (ADR 0003): último valor observado.,- O último período é bom guia do próximo: nao_testavel (hipotese),,1,
1,media_historica,baseline,sim,regra,Baseline obrigatório (ADR 0003): média expansiva do treino.,- Nível aproximadamente constante (estacionária em nível): nao_testavel (hipotese),,1,
2,naive_sazonal,baseline,NÃO,aritmetica,Sem sazonalidade testável em 10 semanas; ciclo de 4 ou 52 semanas não cabe nos dados.,,,0,
3,ets_ses,candidato,sim,regra,"ETS sem tendência nem sazonalidade (ANN), entre o naive e a média.","- Nível aproximadamente constante (estacionária em nível): nao_testavel (hipotese)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): inconclusiva (hipotese)","{'alfa': ('estimado',)}",1,
4,ets_holt_amortecido,candidato,sim,regra,Tendência amortecida (AAdN); não há teste de tendência com 10 pontos.,"- Possível tendência amortecida: nao_testavel (hipotese)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): inconclusiva (hipotese)","{'phi': ('estimado em [0,8; 0,98]',)}",1,
5,arima,candidato,sim,decisao_usuario,"ARIMA mínimo incluído por decisão do usuário, para contraste com as demais opções. Base fraca: estacionariedade e autocorrelação semanais não são testáveis (n = 10 < 30); com 6 pontos no primeiro fold, até 3 parâmetros.",- Estacionariedade (escolha de d): nao_testavel (hipotese)\n- Dependência temporal a modelar: nao_testavel (hipotese),"{'ordem': ((1, 0, 0), (0, 1, 1))}",2,"(1,0,0) com constante; (0,1,1) sem constante. (0,1,1) é próximo do SES. Item C: variante opcional com variável de intervenção na semana 07-19."
6,sarima,candidato,NÃO,aritmetica,"Ciclo sazonal de 4 ou 52 semanas não cabe em 10 pontos. Exclusão sobre o universo do ADR 0003, sinalizada.",,,0,
7,croston,candidato,NÃO,evidencia,"Croston (1972): tamanho ÷ intervalo. Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,
8,sba,candidato,NÃO,evidencia,"Croston com correção de viés (Syntetos–Boylan). Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,
9,tsb,candidato,NÃO,evidencia,"Probabilidade de venda atualizada também nos zeros. Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,


### item D — semanal


,algoritmo,papel,entra,base,motivo,premissas,espaço,comb.,observação
0,naive,baseline,sim,regra,Baseline obrigatório (ADR 0003): último valor observado.,- O último período é bom guia do próximo: nao_testavel (hipotese),,1,
1,media_historica,baseline,sim,regra,Baseline obrigatório (ADR 0003): média expansiva do treino.,- Nível aproximadamente constante (estacionária em nível): nao_testavel (hipotese),,1,
2,naive_sazonal,baseline,NÃO,aritmetica,Sem sazonalidade testável em 10 semanas; ciclo de 4 ou 52 semanas não cabe nos dados.,,,0,
3,ets_ses,candidato,sim,regra,"ETS sem tendência nem sazonalidade (ANN), entre o naive e a média.","- Nível aproximadamente constante (estacionária em nível): nao_testavel (hipotese)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): inconclusiva (hipotese)","{'alfa': ('estimado',)}",1,
4,ets_holt_amortecido,candidato,sim,regra,Tendência amortecida (AAdN); não há teste de tendência com 10 pontos.,"- Possível tendência amortecida: nao_testavel (hipotese)\n- Valores contínuos, erro simétrico (zeros e contagens violam em parte; previsões são truncadas em zero): inconclusiva (hipotese)","{'phi': ('estimado em [0,8; 0,98]',)}",1,
5,arima,candidato,sim,decisao_usuario,"ARIMA mínimo incluído por decisão do usuário, para contraste com as demais opções. Base fraca: estacionariedade e autocorrelação semanais não são testáveis (n = 10 < 30); com 6 pontos no primeiro fold, até 3 parâmetros.",- Estacionariedade (escolha de d): nao_testavel (hipotese)\n- Dependência temporal a modelar: nao_testavel (hipotese),"{'ordem': ((1, 0, 0), (0, 1, 1))}",2,"(1,0,0) com constante; (0,1,1) sem constante. (0,1,1) é próximo do SES. Item C: variante opcional com variável de intervenção na semana 07-19."
6,sarima,candidato,NÃO,aritmetica,"Ciclo sazonal de 4 ou 52 semanas não cabe em 10 pontos. Exclusão sobre o universo do ADR 0003, sinalizada.",,,0,
7,croston,candidato,NÃO,evidencia,"Croston (1972): tamanho ÷ intervalo. Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,
8,sba,candidato,NÃO,evidencia,"Croston com correção de viés (Syntetos–Boylan). Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,
9,tsb,candidato,NÃO,evidencia,"Probabilidade de venda atualizada também nos zeros. Fora: série suave (ADI = 1, 0 zeros em 10 semanas); sem intermitência o método não tem razão de existir.",,,0,


## 3. Resumo: quem entra em cada série

`✓` entra com evidência ou regra; `H` entra como hipótese (sem teste formal que sustente a premissa sazonal); `U` entra por decisão do usuário; `—` não entra.

In [4]:
marca = {
    'evidencia': '✓',
    'regra': '✓',
    'hipotese': 'H',
    'decisao_usuario': 'U',
}
linhas = []
for r in tabela():
    linhas.append({
        'serie': f'{r["sku"]} {r["granularidade"][:3]}',
        'algoritmo': r['algoritmo'],
        'x': marca[r['base']] if r['entra'] else '—',
    })
resumo = (
    pd
    .DataFrame(linhas)
    .pivot(index='algoritmo', columns='serie', values='x')
    .fillna('—')
)
display(resumo)

comb = pd.DataFrame(tabela()).query('entra')
display(
    comb
    .pivot_table(
        index='algoritmo',
        columns=['granularidade', 'sku'],
        values='n_combinacoes',
        aggfunc='first',
    )
    .fillna(0)
    .astype(int)
)
print('máximo de combinações por candidato:', comb['n_combinacoes'].max())
print(
    'candidatos que entram por série:',
    comb.groupby(['sku', 'granularidade']).size().to_dict(),
)

serie,item A dia,item A sem,item B dia,item B sem,item C dia,item C sem,item D dia,item D sem
algoritmo,,,,,,,,
arima,✓,U,✓,U,✓,U,✓,U
croston,✓,—,✓,—,✓,—,✓,—
ets_holt_amortecido,—,✓,—,✓,—,✓,—,✓
ets_holt_winters_aditivo,H,—,✓,—,✓,—,H,—
ets_ses,✓,✓,✓,✓,✓,✓,✓,✓
media_historica,✓,✓,✓,✓,✓,✓,✓,✓
naive,✓,✓,✓,✓,✓,✓,✓,✓
naive_sazonal,H,—,✓,—,✓,—,H,—
prophet,H,—,✓,—,✓,—,H,—


granularidade            diaria                      semanal                \
sku                      item A item B item C item D  item A item B item C   
algoritmo                                                                    
arima                         3      3      3      3       2      2      2   
croston                       3      3      3      3       0      0      0   
ets_holt_amortecido           0      0      0      0       1      1      1   
ets_holt_winters_aditivo      1      1      1      1       0      0      0   
ets_ses                       1      1      1      1       1      1      1   
media_historica               1      1      1      1       1      1      1   
naive                         1      1      1      1       1      1      1   
naive_sazonal                 1      1      1      1       0      0      0   
prophet                       4      4      4      4       0      0      0   
sarima                        3      3      3      3       0      0      0   
sba                           3      3      3      3       0      0      0   
tsb                           3      3      3      3       0      0      0   

granularidade                    
sku                      item D  
algoritmo                        
arima                         2  
croston                       0  
ets_holt_amortecido           1  
ets_holt_winters_aditivo      0  
ets_ses                       1  
media_historica               1  
naive                         1  
naive_sazonal                 0  
prophet                       0  
sarima                        0  
sba                           0  
tsb                           0

máximo de combinações por candidato: 4
candidatos que entram por série: {('item A', 'diaria'): 11, ('item A', 'semanal'): 5, ('item B', 'diaria'): 11, ('item B', 'semanal'): 5, ('item C', 'diaria'): 11, ('item C', 'semanal'): 5, ('item D', 'diaria'): 11, ('item D', 'semanal'): 5}


**Observação (proposta, aguardando aprovação):** na diária entram 11 candidatos em cada série (naive, média, naive sazonal, SES, Holt-Winters aditivo, ARIMA, SARIMA, Croston, SBA, TSB e Prophet); na semanal entram 5 (naive, média, SES, Holt amortecido e ARIMA mínimo). Nenhum candidato passa de 4 combinações. Nas diárias A e D, naive sazonal, Holt-Winters, SARIMA e Prophet entram como **hipótese**.

**Interpretação (proposta):** com 4 a 5 folds de teste e 10 candidatos diários, o risco de escolher por acaso é real; o #7 deve comparar com incerteza e preferir o candidato mais simples em empate. Entrar na tabela não é um indício de bom desempenho.

## 4. Requisito de backtest: item C semanal, semana de 07-19

Decisão do usuário (registro da Fase 1, seção 3.5, alternativa 1): manter o valor (40 unidades) e relatar a sensibilidade das métricas **com e sem essa semana** e **com e sem os folds 1 e 2**. Esta seção só mostra os dados; nenhuma métrica de modelo é calculada.

In [5]:
fc = folds[(folds.sku == 'item C') & (folds.granularidade == 'semanal')]
display(fc)
serie_c = w[w.sku == 'item C'].set_index('semana')['quantidade']
display(serie_c.to_frame('item C, semanal'))
sem = pd.Timestamp('2021-07-19')
teste_do_fold = fc[
    (pd.to_datetime(fc.teste_inicio) <= sem)
    & (pd.to_datetime(fc.teste_fim) >= sem)
]
display(teste_do_fold)
print(
    'mediana das outras 9 semanas:',
    serie_c.drop(sem).median(),
    '| máximo das outras:',
    serie_c.drop(sem).max(),
)

,sku,granularidade,fold,treino_inicio,treino_fim,teste_inicio,teste_fim
22,item C,semanal,1,2021-06-07,2021-07-18,2021-07-19,2021-07-25
23,item C,semanal,2,2021-06-07,2021-07-25,2021-07-26,2021-08-01
24,item C,semanal,3,2021-06-07,2021-08-01,2021-08-02,2021-08-08
25,item C,semanal,4,2021-06-07,2021-08-08,2021-08-09,2021-08-15


,"item C, semanal"
semana,
2021-06-07,9.0
2021-06-14,4.0
2021-06-21,16.0
2021-06-28,16.0
2021-07-05,19.0
2021-07-12,18.0
2021-07-19,40.0
2021-07-26,17.0
2021-08-02,19.0


,sku,granularidade,fold,treino_inicio,treino_fim,teste_inicio,teste_fim
22,item C,semanal,1,2021-06-07,2021-07-18,2021-07-19,2021-07-25


mediana das outras 9 semanas: 16.0 | máximo das outras: 19.0


**Observação (proposta, aguardando aprovação):** a semana 07-19 (40 unidades) é o teste do fold 1 do item C semanal; nas outras 9 semanas o valor vai de 4 a 19, com mediana 16,0 (a mediana de 16,5 citada no registro da Fase 1, seção 3.5, é a das 10 semanas, com a de 07-19 incluída). Os folds 2 a 4 têm a semana 07-19 no treino.

**Requisito registrado para o #7 (item C semanal):** calcular as métricas (i) com todas as semanas de teste, (ii) sem a semana 07-19 e (iii) sem os folds 1 e 2; reportar a diferença. ARIMA mínimo pode ter a variante com variável de intervenção na semana 07-19; métricas robustas (mediana dos erros) são complemento opcional. Nenhuma alteração da série.

## 5. Limitações e o que ainda não se sabe

- A tabela é uma decisão de **entrada**, não de desempenho: nada foi ajustado. Hipóteses H1 a H5 do plano são para o #7 em diante.
- Os testes do #5 são exploratórios (n = 76 a 79; 10 semanas), com premissas aproximadas; "confirmada" e "inconclusiva" descrevem o resultado dos testes, não prova.
- O ARIMA semanal entra por decisão do usuário, com base fraca; SARIMA semanal sai por aritmética, o que é uma restrição sobre o universo do ADR 0003 (sinalizada, o ADR não foi alterado).
- Venda é proxy de demanda; ruptura aparece como zero e não é distinguível.
- Opção A (ticket #4): a diária contém 16/08 a 18/08, dias da semana OOT de 16/08; as decisões da semanal usam só a visão semanal.